# Getting started with the CEY20 v2 California exposure reference year

Hourly 365-day reference year (temperature, dew point, surface pressure, relative humidity) for 711 ERA5-Land points (0.25°) covering California. Each month comes from one actual source year (2006–2025) chosen per point. This notebook loads the point file and a county file and shows two quick views. See `README.md` for definitions and limitations (exposure reference year, not an energy TMY; no radiation, no wind).

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Kaggle input, or ./data after `python -m cey20ca download` (run from the repository root or notebooks/)
candidates = (glob.glob('/kaggle/input/**/cey20_v2_california_points.nc', recursive=True)
              + glob.glob('data/cey20_v2_california_points.nc') + glob.glob('../data/cey20_v2_california_points.nc'))
root = os.path.dirname(candidates[0])
print(root)
print(sorted(os.listdir(root)))

## All points: NetCDF
`hour` = 0–8759 of a 365-day UTC year (29 February removed); `month` = 1–12.

In [ ]:
ds = xr.open_dataset(os.path.join(root, 'cey20_v2_california_points.nc'))
ds

Selected source year per month for the first five points:

In [ ]:
pd.DataFrame(ds['selected_source_year'].values[:, :5], index=range(1, 13),
             columns=[f"point {int(i)}" for i in ds['product_point_index'].values[:5]])

## One county: hourly CSV
The first two lines start with `#` (county, centroid, chosen point, distance).

In [ ]:
index = pd.read_csv(os.path.join(root, 'counties_index.csv'), dtype={'geoid': str})
row = index.loc[index['name'] == 'Los Angeles'].iloc[0]
# Kaggle unpacked counties.zip one folder deeper than counties_index.csv says: find the file by name
county_path = glob.glob(os.path.join(root, '**', os.path.basename(row['file'])), recursive=True)[0]
county = pd.read_csv(county_path, comment='#')
print(row[['name', 'point_lat', 'point_lon', 'distance_km']].to_dict())
county.head()

In [ ]:
monthly = county.groupby('month')[['temperature_c', 'rh_pct']].mean()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(monthly.index, monthly['temperature_c'], marker='o', linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xlabel('Month')
ax.set_ylabel('Mean 2 m temperature (°C)')
ax.set_title(f"{row['name']} County point: monthly mean temperature of the reference year")
ax.grid(alpha=0.3)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
plt.tight_layout()

## Descriptive metrics (not a validation)
Absolute relative error of annual hours with RH ≥ 80 % against the mean of the 20 actual years. Large relative errors occur mostly where the reference count is small (southeastern deserts).

In [ ]:
pm = pd.read_csv(os.path.join(root, 'point_metrics.csv'))
err = pd.to_numeric(pm['hours_rh_ge_80_abs_rel_error'], errors='coerce')
fig, ax = plt.subplots(figsize=(6, 6.5))
sc = ax.scatter(pm['longitude'], pm['latitude'], c=err * 100, cmap='Blues', vmin=0, vmax=50, s=18,
                edgecolors='#9a9a9a', linewidths=0.4)
fig.colorbar(sc, ax=ax, label='Absolute relative error, RH ≥ 80 % hours (%)  (capped at 50)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect(1 / np.cos(np.deg2rad(37)))
ax.set_title('Reference year vs. 20-year mean (descriptive)')
plt.tight_layout()
print('within ±5 %:', int(pm['hours_rh_ge_80_within_5pct'].astype(str).str.lower().eq('true').sum()), 'of', len(pm), 'points')

## Citation
Okley, W. (2026): CEY20 v2 exposure reference year: California (0.25 degree), dataset v1.0.0. Zenodo. https://doi.org/10.5281/zenodo.24016356 — Contains modified Copernicus Climate Change Service information (2026).